In [6]:
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0"
}

def scrape_page(page):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    try:
        response = requests.get(url, headers=headers, timeout=10)

        soup = BeautifulSoup(response.text, "html.parser")

        books = soup.find_all("article", class_="product_pod")

        data = []

        for book in books:

            title = book.h3.a["title"]

            price = book.find("p", class_="price_color").text

            availability = book.find(
                "p",
                class_="instock availability"
            ).text.strip()

            data.append({
                "Title": title,
                "Price": price,
                "Availability": availability
            })

        return data

    except Exception as e:
        print(f"Error on page {page}: {e}")
        return []

all_books = []

with ThreadPoolExecutor(max_workers=10) as executor:

    results = executor.map(scrape_page, range(1, 6))

    for result in results:
        all_books.extend(result)

df = pd.DataFrame(all_books)

print(df.head())

df.to_csv("books.csv", index=False)

print("Saved Successfully")

                                   Title    Price Availability
0                   A Light in the Attic  Â£51.77     In stock
1                     Tipping the Velvet  Â£53.74     In stock
2                             Soumission  Â£50.10     In stock
3                          Sharp Objects  Â£47.82     In stock
4  Sapiens: A Brief History of Humankind  Â£54.23     In stock
Saved Successfully


/usr/local/lib/python3.12/dist-packages/bs4/__init__.py:870: RuntimeWarning: coroutine 'main' was never awaited
  def object_was_parsed(


In [11]:
all_books

[{'Title': 'A Light in the Attic',
  'Price': 'Â£51.77',
  'Availability': 'In stock'},
 {'Title': 'Tipping the Velvet',
  'Price': 'Â£53.74',
  'Availability': 'In stock'},
 {'Title': 'Soumission', 'Price': 'Â£50.10', 'Availability': 'In stock'},
 {'Title': 'Sharp Objects', 'Price': 'Â£47.82', 'Availability': 'In stock'},
 {'Title': 'Sapiens: A Brief History of Humankind',
  'Price': 'Â£54.23',
  'Availability': 'In stock'},
 {'Title': 'The Requiem Red', 'Price': 'Â£22.65', 'Availability': 'In stock'},
 {'Title': 'The Dirty Little Secrets of Getting Your Dream Job',
  'Price': 'Â£33.34',
  'Availability': 'In stock'},
 {'Title': 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull',
  'Price': 'Â£17.93',
  'Availability': 'In stock'},
 {'Title': 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics',
  'Price': 'Â£22.60',
  'Availability': 'In stock'},
 {'Title': 'The Black Maria', 'Price': 'Â£52.15', 'Avail

Example 2 — Ultra Fast Async Scraper

This teaches:

asyncio
aiohttp
async scraping

In [1]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0"
}

async def scrape_page(session, page):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    try:
        async with session.get(url, headers=headers) as response:

            html = await response.text()

            soup = BeautifulSoup(html, "html.parser")

            books = soup.find_all("article", class_="product_pod")

            data = []

            for book in books:

                title = book.h3.a["title"]

                price = book.find(
                    "p",
                    class_="price_color"
                ).text

                availability = book.find(
                    "p",
                    class_="instock availability"
                ).text.strip()

                data.append({
                    "Title": title,
                    "Price": price,
                    "Availability": availability
                })

            return data

    except Exception as e:
        print(f"Error: {e}")
        return []

async def main():

    connector = aiohttp.TCPConnector(limit=10)

    async with aiohttp.ClientSession(
        connector=connector
    ) as session:

        tasks = []

        for page in range(1, 6):
            tasks.append(scrape_page(session, page))

        results = await asyncio.gather(*tasks)

        all_books = []

        for result in results:
            all_books.extend(result)

        df = pd.DataFrame(all_books)

        print(df.head())

        df.to_csv("books_async.csv", index=False)

        print("Saved Successfully")

await main()

                                   Title   Price Availability
0                   A Light in the Attic  £51.77     In stock
1                     Tipping the Velvet  £53.74     In stock
2                             Soumission  £50.10     In stock
3                          Sharp Objects  £47.82     In stock
4  Sapiens: A Brief History of Humankind  £54.23     In stock
Saved Successfully
